# TaaS Core Studio — Colab T4 deployment

**Instruction guide:** follow `docs/deployment/COLAB_T4_QUICKSTART.md` before running this notebook. The guide explains the Colab runtime selection, required Secrets, ModelProfile upload, launch, browser access, troubleshooting and shutdown.

This notebook turns the Colab runtime into the host for the TaaS Studio web application. It installs the pinned TaaS source, checks the allocated GPU, starts the TaaS API/UI, and verifies readiness.

**Runtime:** select `T4 GPU` in Runtime → Change runtime type before running. GPU availability is decided by Colab, not by this notebook.

**Security:** owner access uses Colab's notebook port proxy. Optional team access requires a named Cloudflare Tunnel and a Cloudflare Access application/policy protecting the hostname. A random `trycloudflare.com` URL alone is not authentication. Public access is disabled by default.

**Current product scope:** this release provides GoalSpec/tool metadata input and TaaS discovery results. MCP server invocation, agent orchestration, live execution stream, and durable run database are not implemented yet.

In [ ]:
import os, sys, subprocess, pathlib, json, time, shutil, socket, urllib.request, urllib.error
from IPython.display import display, HTML

REPO_URL = "https://github.com/msellamiTN/taas-provenance-discovery.git"
TAAS_COMMIT = "3f74c753a9a93629ecee2718620fb06bd3affc82"  # reviewed Studio UI/API revision
WORKDIR = pathlib.Path('/content/taas-provenance-discovery')
APP_HOST = '127.0.0.1'
APP_PORT = 8000

if not WORKDIR.exists():
    subprocess.run(['git','clone',REPO_URL,str(WORKDIR)], check=True)
subprocess.run(['git','fetch','--all'], cwd=WORKDIR, check=True)
subprocess.run(['git','checkout','--detach',TAAS_COMMIT], cwd=WORKDIR, check=True)
actual = subprocess.check_output(['git','rev-parse','HEAD'], cwd=WORKDIR, text=True).strip()
assert actual == TAAS_COMMIT, (actual, TAAS_COMMIT)
print('Pinned TaaS commit:', actual)

In [ ]:
## Step 2 — Check the allocated GPU
Run this cell and confirm that `nvidia-smi` reports an NVIDIA Tesla T4. If it reports no GPU, stop here and select Runtime → Change runtime type → T4 GPU (if available), then reconnect and restart the notebook. The web application can run on CPU, but local GPU model inference will not be available.

# Inspect the actual Colab hardware. Do not infer GPU availability from the runtime label.
gpu_check = subprocess.run(['bash','-lc','nvidia-smi'], capture_output=True, text=True)
GPU_AVAILABLE = gpu_check.returncode == 0
if GPU_AVAILABLE:
    print(gpu_check.stdout)
else:
    print('No NVIDIA GPU visible. Studio will run CPU-only; enable a T4 runtime to use GPU-backed models.')
print('Python:', sys.version.split()[0])
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES','(not set)'))

## Install TaaS Studio

The current application is a FastAPI backend with a same-origin browser UI. Installation below uses the repository's declared `studio` optional dependencies.

In [ ]:
## Step 3 — Install TaaS Studio
Run this cell once after the repository checkout. It installs the optional FastAPI/Uvicorn dependencies and verifies that the API module imports. If installation fails, inspect the first pip error, restart the Colab runtime, and rerun from Step 1.

subprocess.run([sys.executable,'-m','pip','install','--upgrade','pip'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.[studio]'], cwd=WORKDIR, check=True)
subprocess.run([sys.executable,'-c','import fastapi, uvicorn, taas.studio.api; print("Studio imports OK")'], cwd=WORKDIR, check=True)

## Configure credentials and model profile

Create Colab Secrets named `TAAS_STUDIO_API_TOKEN` and (for model use) any provider-specific keys required by your ModelProfile. Upload a reviewed ModelProfile JSON. Secret values are not printed or saved in the notebook. The current discovery API requires a profile, even when the inference provider is external.

In [ ]:
## Step 4 — Add credentials and ModelProfile
Before running this cell, open the Colab key icon (Secrets) and add `TAAS_STUDIO_API_TOKEN`; grant this notebook access. Use a long random token. Then run the cell and upload exactly one reviewed ModelProfile JSON when prompted. Do not put provider API keys inside that JSON or print secret values. If you do not have a valid TaaS ModelProfile yet, stop and prepare one from the repository's model-profile examples/documentation before continuing.

from google.colab import userdata, files

api_token = userdata.get('TAAS_STUDIO_API_TOKEN')
if not api_token or len(api_token) < 24:
    raise RuntimeError('Set a strong TAAS_STUDIO_API_TOKEN (at least 24 characters) in Colab Secrets.')
os.environ['TAAS_STUDIO_API_TOKEN'] = api_token

uploaded = files.upload()
json_files = [name for name in uploaded if name.lower().endswith('.json')]
if len(json_files) != 1:
    raise RuntimeError('Upload exactly one reviewed ModelProfile JSON file.')
profile_path = pathlib.Path('/content') / json_files[0]
profile_obj = json.loads(profile_path.read_text(encoding='utf-8'))
if not isinstance(profile_obj, dict):
    raise ValueError('ModelProfile must be a JSON object.')
os.environ['TAAS_MODEL_PROFILE'] = str(profile_path.resolve())
print('TaaS API credential and ModelProfile configured.')

## Optional: local model backend on the T4

TaaS's discovery engine remains CPU-oriented. The GPU is useful when an actual model backend (local LLM, embedding model, or reranker) is started and configured. This optional section uses an OpenAI-compatible local endpoint contract. Supply a model-serving command that you have validated for this Colab image and GPU; the notebook does not install an arbitrary model server or download a large model by default.

Set `TAAS_LOCAL_MODEL_BASE_URL` only after you have successfully started the model server. For example, an OpenAI-compatible endpoint often exposes `/v1`; configure the matching model/profile on the TaaS side. A T4 is typically memory-constrained, so select a quantized model appropriate to its available VRAM.

In [ ]:
# Optional model endpoint configuration. Leave unset to use the provider configured in ModelProfile.
LOCAL_MODEL_BASE_URL = os.environ.get('TAAS_LOCAL_MODEL_BASE_URL','').strip()
if LOCAL_MODEL_BASE_URL:
    print('Configured local model endpoint:', LOCAL_MODEL_BASE_URL)
else:
    print('No local model endpoint configured; Studio will use its server-side ModelProfile configuration.')

## Step 6 — Start TaaS Studio
Run this cell to launch Uvicorn inside the Colab VM. It binds to `127.0.0.1`, waits for `/health` and `/`, and stops with a clear error if startup fails. Keep this cell running while using the Studio UI.

# Stop a previous notebook-started server, if this cell is rerun.
try:
    if 'studio_process' in globals() and studio_process.poll() is None:
        studio_process.terminate()
        studio_process.wait(timeout=10)
except Exception:
    pass

studio_log_path = pathlib.Path('/content/taas-studio.log')
studio_log = studio_log_path.open('w')
studio_process = subprocess.Popen(
    [sys.executable,'-m','uvicorn','taas.studio.api:app','--host',APP_HOST,'--port',str(APP_PORT)],
    cwd=WORKDIR, env=os.environ.copy(), stdout=studio_log, stderr=subprocess.STDOUT
)

def wait_http(url, attempts=30):
    last = None
    for _ in range(attempts):
        if studio_process.poll() is not None:
            studio_log.flush()
            raise RuntimeError('TaaS Studio exited. Inspect /content/taas-studio.log')
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                return response.status, response.read().decode('utf-8')
        except Exception as exc:
            last = exc
            time.sleep(1)
    raise RuntimeError(f'Studio endpoint did not become ready: {last}')

print('Studio health:', wait_http(f'http://{APP_HOST}:{APP_PORT}/health'))
print('Studio UI:', wait_http(f'http://{APP_HOST}:{APP_PORT}/')[0])

In [ ]:
from google.colab import output
display(HTML('<p><b>TaaS Core Studio is running.</b> Click the proxied link below.</p>'))
proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({APP_PORT})")
display(HTML(f'<p><a href="{proxy_url}" target="_blank" rel="noopener">Open TaaS Core Studio</a></p>'))
print('The link is scoped to this Colab runtime/session:', proxy_url)

## Step 7 — Open the Studio in your browser
Run this cell after Step 6. Click the generated link. If the link does not open, rerun the cell, verify the Colab runtime is still connected, and confirm that the API health check still responds. Do not copy/share this owner-only proxied URL as a permanent deployment address.

from google.colab import output
display(HTML('<p><b>TaaS Core Studio is running.</b> Click the proxied link below.</p>'))
proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({APP_PORT})")
display(HTML(f'<p><a href="{proxy_url}" target="_blank" rel="noopener">Open TaaS Core Studio</a></p>'))
print('The link is scoped to this Colab runtime/session:', proxy_url)

In [ ]:
## Optional — Protected team access

Skip this section for personal use. For team sharing, first create a named Cloudflare Tunnel and a Cloudflare Access application/policy for a hostname you control. Configure the tunnel origin as `http://127.0.0.1:8000`, add `CF_TUNNEL_TOKEN` in Colab Secrets, then set `enable_cf = True` in the next cell. The Access policy must protect the complete hostname. A tunnel token alone is not user authentication. Do not use a quick tunnel for sensitive work.

## Optional protected public access with Cloudflare Tunnel

Use this only after creating a Cloudflare Access application and policy for a hostname you control. Add the tunnel token as a Colab Secret named `CF_TUNNEL_TOKEN`. Configure that tunnel in Cloudflare to route the protected hostname to `http://127.0.0.1:8000`. A tunnel token alone creates connectivity; **Cloudflare Access policy** is what restricts who can visit the UI. Do not use a `trycloudflare.com` quick tunnel for this authenticated Studio.

This cell uses the Cloudflare-managed tunnel token and never prints it. The actual URL should be your preconfigured Access hostname.

In [ ]:
enable_cf = False  # Set True only after Cloudflare Access + named tunnel are configured.
if enable_cf:
    cf_token = userdata.get('CF_TUNNEL_TOKEN')
    if not cf_token:
        raise RuntimeError('Add CF_TUNNEL_TOKEN to Colab Secrets.')
    cloudflared = shutil.which('cloudflared')
    if not cloudflared:
        subprocess.run(['bash','-lc','wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared'], check=True)
        cloudflared = '/usr/local/bin/cloudflared'
    cf_log = open('/content/cloudflared.log','w')
    cf_process = subprocess.Popen([cloudflared,'tunnel','run','--token',cf_token], stdout=cf_log, stderr=subprocess.STDOUT)
    print('Cloudflare tunnel process started. Verify your configured Access hostname and policy in Cloudflare dashboard before sharing it.')
else:
    print('Public tunnel disabled. Use the Colab-proxied URL above.')

## Stop the Studio

Run this when finished. Colab may still terminate the runtime automatically. Do not rely on its local filesystem for persistent experiment results.

In [ ]:
try:
    if 'cf_process' in globals() and cf_process.poll() is None:
        cf_process.terminate()
except Exception:
    pass
if studio_process.poll() is None:
    studio_process.terminate()
    studio_process.wait(timeout=10)
studio_log.close()
print('TaaS Studio stopped.')